In [1]:
import pandas as pd
from sklearn.metrics import accuracy_score, f1_score, classification_report

df = pd.read_csv('golden_eval_with_llm_predictions.csv')
y_true = df['label_intent'].tolist()
y_pred = df['llm_intent'].tolist()

acc = accuracy_score(y_true, y_pred)
f1 = f1_score(y_true, y_pred, average='macro', zero_division=0)

print('=== LLM few-shot classifier (llama3.1:8b via Ollama) ===')
print(f'accuracy: {acc:.3f}')
print(f'macro F1: {f1:.3f}')
print()
print(classification_report(y_true, y_pred, zero_division=0))

=== LLM few-shot classifier (llama3.1:8b via Ollama) ===
accuracy: 0.688
macro F1: 0.675

                               precision    recall  f1-score   support

      account_access_security       0.85      0.73      0.79        15
billing_subscription_purchase       0.83      0.83      0.83        12
                 connectivity       0.88      0.47      0.61        15
             data_backup_loss       0.81      0.87      0.84        15
               feature_how_to       0.40      0.93      0.56        15
            hardware_physical       0.69      0.60      0.64        15
                other_unclear       0.35      0.40      0.38        15
  software_update_performance       0.83      0.69      0.75        71

                     accuracy                           0.69       173
                    macro avg       0.71      0.69      0.67       173
                 weighted avg       0.74      0.69      0.70       173



In [3]:
y_true = df['label_intent'].tolist()

results = {
    'Baseline 1: Majority class': (0.410, 0.073),
    'Baseline 2: TF-IDF + LogReg': (0.503, 0.342),
    'LLM few-shot (llama3.1:8b)': (
        accuracy_score(y_true, df['llm_intent'].tolist()),
        f1_score(y_true, df['llm_intent'].tolist(), average='macro', zero_division=0)
    ),
}

print('=== Results ===')
print('-'*56)
for name, (acc, f1) in results.items():
    print(f'{name:<32}{acc:<12.3f}{f1:<12.3f}')

=== Results ===
--------------------------------------------------------
Baseline 1: Majority class      0.410       0.073       
Baseline 2: TF-IDF + LogReg     0.503       0.342       
LLM few-shot (llama3.1:8b)      0.688       0.675       


In [4]:
# confusion pairs - what gets confused with what
wrong = df[df['label_intent'] != df['llm_intent']]
print(f'Wrong: {len(wrong)}/{len(df)} ({100*len(wrong)/len(df):.1f}%)')
print()
print('Most common confusion pairs (true -> predicted):')
pairs = wrong.groupby(['label_intent','llm_intent']).size().sort_values(ascending=False)
print(pairs.head(15))

Wrong: 54/173 (31.2%)

Most common confusion pairs (true -> predicted):
label_intent                   llm_intent                   
software_update_performance    feature_how_to                   15
                               other_unclear                     4
other_unclear                  software_update_performance       4
hardware_physical              other_unclear                     4
account_access_security        other_unclear                     3
other_unclear                  feature_how_to                    3
connectivity                   software_update_performance       3
                               account_access_security           2
billing_subscription_purchase  data_backup_loss                  2
other_unclear                  hardware_physical                 2
connectivity                   feature_how_to                    2
account_access_security        software_update_performance       1
data_backup_loss               feature_how_to                  

In [5]:
confused = df[(df['label_intent']=='software_update_performance') & (df['llm_intent']=='feature_how_to')]
for _, row in confused.head(15).iterrows():
    print('-', row['opening_text'][:150])

- @AppleSupport Why are the buttons like that? https://t.co/XHgG77luaR
- @AppleSupport Not an issue but when u put a new SIM, go to settings-&gt;AppleID. It asks you to “Uptade” instead of “Update” your trusted phone.
- What does that mean?? I’m not trying to send anything... please @6717 @115858 @46228 @118936 https://t.co/EHz1jaDGHM
- @AppleSupport why didn't I get my monthly award for closing my rings each day?
- @AppleSupport Y’all gotta do better with this keyboard situation is still messed up 😒
- Anybody else experiencing the @AppleSupport bug where the letter I turning into a weird set of glyphs? #Igate
- Checking snapchat stops my music now &amp; that is not ok ☹️ @917 @115858
- .@AppleSupport iOS11 - when playing podcasts, the lock screen doesn’t show correct play times until paused. Could you log a fault? Thanks.
- @AppleSupport Hi. I have an apple 7 plus. Same problem. When im video call, I can not manage the volume while talking. Can you tell me . what is the p
- How come e

In [6]:
print('=== true=hardware_physical, predicted=other_unclear ===')
sub = df[(df['label_intent']=='hardware_physical') & (df['llm_intent']=='other_unclear')]
for _, row in sub.iterrows():
    print('-', row['opening_text'][:150])
print()
print('=== true=other_unclear, predicted=software_update_performance ===')
sub = df[(df['label_intent']=='other_unclear') & (df['llm_intent']=='software_update_performance')]
for _, row in sub.iterrows():
    print('-', row['opening_text'][:150])

=== true=hardware_physical, predicted=other_unclear ===
- @AppleSupport
sometimes my iphone hot when not connected to the internet :(
- Had the misfortune of using my @115858 headphones for the first time and boy was i disappointed. The ones at Dadar station work better.
- @AppleSupport is my 2 week old Apple Watch sports strap supposed to look like this two weeks after getting it? https://t.co/h9ocOcHX3z
- anybody else’s phone touchscreen absolutely horrible rn?

=== true=other_unclear, predicted=software_update_performance ===
- Am I the only one having issues with #iOS11? @115858 @AppleSupport
- @AppleSupport Sent you a DM about issues I'm having with the new iOS.
- @115858 thanks for making my phone suck donkeys nerds. You’re intentionally trying to force everyone to buy the X by f’ng with the software. Dicks.
- Answer my this. Why the fuck my phone been tweaking! @115858


In [8]:
# We don't have an explicit LLM escalation decision yet -- only intent.
# For now, derive it via the DEFAULT_ESCALATE mapping per predicted intent,
# to see how far a pure intent->escalation-default policy gets us.
DEFAULT_ESCALATE = {
    'software_update_performance': False,
    'feature_how_to': False,
    'connectivity': False,
    'hardware_physical': True,
    'account_access_security': True,
    'data_backup_loss': True,
    'billing_subscription_purchase': True,
    'other_unclear': False,
}
df['llm_escalate_derived'] = df['llm_intent'].map(DEFAULT_ESCALATE)
df['label_escalate'] = df['label_escalate'].astype(bool)

from sklearn.metrics import precision_score, recall_score, f1_score, confusion_matrix

y_true = df['label_escalate']
y_pred = df['llm_escalate_derived']

prec = precision_score(y_true, y_pred)
rec = recall_score(y_true, y_pred)
f1 = f1_score(y_true, y_pred)
cm = confusion_matrix(y_true, y_pred)

print('Escalation decision (derived from predicted intent defaults):')
print(f'  precision: {prec:.3f}')
print(f'  recall:    {rec:.3f}')
print(f'  f1:        {f1:.3f}')
print()
print('Confusion matrix [rows=true, cols=pred], order=[False,True]:')
print(cm)
print()
false_negatives = df[(df['label_escalate']==True) & (df['llm_escalate_derived']==False)]
print(f'FALSE NEGATIVES (should escalate, did not) -- the dangerous direction: {len(false_negatives)}')
for _, row in false_negatives.iterrows():
    print(
        f"true={row['label_intent']:<28} "
        f"pred={row['llm_intent']:<28} | "
        f"{row['opening_text'][:100]}"
    )

Escalation decision (derived from predicted intent defaults):
  precision: 0.815
  recall:    0.721
  f1:        0.765

Confusion matrix [rows=true, cols=pred], order=[False,True]:
[[102  10]
 [ 17  44]]

FALSE NEGATIVES (should escalate, did not) -- the dangerous direction: 17
true=hardware_physical            pred=other_unclear                | @AppleSupport
sometimes my iphone hot when not connected to the internet :(
true=software_update_performance  pred=software_update_performance  | @115858 your updates since the release of the iPhone 8 and x have SCREWED my phone up. Videos don’t 
true=software_update_performance  pred=software_update_performance  | @115858 My iPhone has been extremely slow ever since I updated my phone. It will constantly freeze a
true=software_update_performance  pred=software_update_performance  | @115858 @AppleSupport after upgrading to the latest #iOS 11.1.2 on my iPad, going back the home scre
true=account_access_security      pred=other_unclear          

In [9]:
DEFAULT_ESCALATE = {
    'software_update_performance': False, 'feature_how_to': False, 'connectivity': False,
    'hardware_physical': True, 'account_access_security': True, 'data_backup_loss': True,
    'billing_subscription_purchase': True, 'other_unclear': False,
}
df['llm_escalate_derived'] = df['llm_intent'].map(DEFAULT_ESCALATE)
df.to_csv('golden_eval_final_scored.csv', index=False)
print('saved, shape:', df.shape)

saved, shape: (173, 20)
